## Flexcube
Goal: To load the relevant Flexcube data into SIRA data model

###### <span style="color:orange">Authors:</span>
1.  Ruud van Laar

###### <span style="color:orange">Business value:</span>
1.	For FEC-related insights into CBT Flexcube transactions accross the W&R client scope. Focus on counterparty geographical risks.


###### <span style="color:blue">Example Flow of the logic:</span>
1.  Load and connect to data (create a more specific md cell at the start of that step)
    - All countries Transactions
    - All countries Products
2.  Transform data (create a more specific md cell at that stage)
3.  final step exporting data (create a more specific md cell at that stage)
    - export to parquet / deta for strategic
    - export to DWH for tactical

In [0]:
# generic imports
import pandas as pd
from notebookutils import mssparkutils
from datetime import datetime
from datetime import timedelta
import com.microsoft.spark.sqlanalytics
from com.microsoft.spark.sqlanalytics.Constants import Constants
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql.functions import col,lit

currentDate = datetime.today().strftime('%Y%m%d')
Yesterdate = (datetime.today() - timedelta(1)).strftime('%Y%m%d')

currentYear = datetime.today().strftime('%Y')
currentMonth = datetime.today().strftime('%m')
currentDay = datetime.today().strftime('%d')

In [0]:
currentDate = datetime.today().strftime('%Y%m%d')
Yesterdate = (datetime.today() - timedelta(1)).strftime('%Y%m%d')

load_dts = 'BUSINESS_DTS=' + Yesterdate + 'T000000Z'

In [0]:
# example of loading files from GDP

# example of loading data from synapse (?)


In [0]:
# Flexcube CBT Collibra: https://rabobank.collibra.com/asset/f254d8d9-86cb-42f4-a751-0d4adeeffc1f
# TX:  https://edlcorestdeuprod0001.dfs.core.windows.net/core-cbt/transaction_CAT/100/data/BUSINESS_DTS=20240128T000000Z


spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_CAT' + '/100/data/' + load_dts + '/*.parquet', format='parquet').createOrReplaceTempView('transaction_CAT')

In [0]:
%sql
select * from transaction_CAT limit 3

In [0]:
countrylist = ['CNS', 'INM', 'USN', 'NLU', 'HKG', 'CAT', 'SGP']

In [0]:
for country in countrylist:
    spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_' + country + '/100/data/' + 'BUSINESS_DTS=' + '202401*/*.parquet', format='parquet').createOrReplaceTempView('tx_jan_'+country)

In [0]:
for country in countrylist:
    c = spark.sql(f"Select count(*) from tx_jan_{country}").collect()
    print(c)


In [0]:
%sql
select * from tx_jan_USN limit 3

In [0]:
additionalSettlementColumns = [
    "additionalSettlement_accountBranchName",
    "additionalSettlement_accountBranch",
    "additionalSettlement_accountCurrency",
    "additionalSettlement_accountDescription",
    "additionalSettlement_accountNumber",
    "additionalSettlement_amountTagDescription",
    "additionalSettlement_amountTagType",
    "additionalSettlement_beneficiaryInstitutionCountryDescription",
    "additionalSettlement_beneficiaryInstitutionOneDescription",
    "additionalSettlement_correspondentOneDescription",
    "additionalSettlement_debitOrCreditIndicator",
    "additionalSettlement_intermediaryOneDescription",
    "additionalSettlement_orderingInstitutionOneDescription",
    "additionalSettlement_reimburseInstitutionOneDescription",
    "additionalSettlement_settlementAmount",
    "additionalSettlement_tagCurrencyDescription"
]

advices = ["advices_receiverReference"]

incomingMessage = ["incomingMessage_senderName",
    "incomingMessage_senderReference",
    "incomingMessage_swiftMessageType",
    "incomingMessage_swiftMessageTypeDescription"]

outgoingMessage = ["outgoingMessage_receiverName",
    "outgoingMessage_receiverReference",
    "outgoingMessage_swiftMessageType",
    "outgoingMessage_swiftMessageTypeDescription"]

In [0]:
dbutils.fs.ls("abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/transaction_CNS/100/data/")

In [0]:
# Loading all historic files for the current and last calendar year
#load_dts= 'BUSINESS_DTS=202{3,4}*T000000Z'
load_dts = 'BUSINESS_DTS=20240221*'

df_transaction_CNS_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_CNS' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_INM_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_INM' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_USN_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_USN' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_NLU_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_NLU' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_HKG_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_HKG' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_CAT_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_CAT' + '/100/data/' + load_dts + '/*.parquet', format='parquet')
df_transaction_SGP_Hist = spark.read.load('abfss://core-cbt@edlcorestdeuprod0001.dfs.core.windows.net/'+'transaction_SGP' + '/100/data/' + load_dts + '/*.parquet', format='parquet')

In [0]:
countrylist

In [0]:
# unioning all transaction files
df_transaction_ALL = df_transaction_CNS_Hist.union(df_transaction_INM_Hist).union(df_transaction_USN_Hist).union(df_transaction_NLU_Hist).union(df_transaction_HKG_Hist).union(df_transaction_CAT_Hist).union(df_transaction_SGP_Hist)

In [0]:
df_transaction_ALL.createOrReplaceTempView("df_ALL")

uniquecompare = spark.sql("""select count(*) as tot, count(distinct(TransactionReferenceCode)) as trc, count(distinct(settlement_endtoEndTransactionReference)) as etetr from df_ALL""")

# TransactionReferenceCode
display(uniquecompare)


In [0]:
#printing input to use for the query in a sec.

# list of columns to do array_zip over.
additionalSettlementColumnsString = '"' + '", "'.join(additionalSettlementColumns) + '"'
print('"' + '", "'.join(additionalSettlementColumns) + '" \n')

#using python to create the code to run next:
for col in additionalSettlementColumns:
    print(f'        F.col("new.{col}").alias("{col}"),\\')

In [0]:
df_ALL_additionalsettlement = df_transaction_ALL.withColumn("new", F.arrays_zip('additionalSettlement_accountBranchName', 'additionalSettlement_accountBranch', 'additionalSettlement_accountCurrency', 'additionalSettlement_accountDescription', 'additionalSettlement_accountNumber', 'additionalSettlement_amountTagDescription', 'additionalSettlement_amountTagType', 'additionalSettlement_beneficiaryInstitutionCountryDescription', 'additionalSettlement_beneficiaryInstitutionOneDescription', 'additionalSettlement_correspondentOneDescription', 'additionalSettlement_debitOrCreditIndicator', 'additionalSettlement_intermediaryOneDescription', 'additionalSettlement_orderingInstitutionOneDescription', 'additionalSettlement_reimburseInstitutionOneDescription', 'additionalSettlement_settlementAmount', 'additionalSettlement_tagCurrencyDescription'))\
    .withColumn("new", F.explode("new"))\
    .select('TransactionReferenceCode', \
        F.col("new.additionalSettlement_accountBranchName").alias("additionalSettlement_accountBranchName"),\
        F.col("new.additionalSettlement_accountBranch").alias("additionalSettlement_accountBranch"),\
        F.col("new.additionalSettlement_accountCurrency").alias("additionalSettlement_accountCurrency"),\
        F.col("new.additionalSettlement_accountDescription").alias("additionalSettlement_accountDescription"),\
        F.col("new.additionalSettlement_accountNumber").alias("additionalSettlement_accountNumber"),\
        F.col("new.additionalSettlement_amountTagDescription").alias("additionalSettlement_amountTagDescription"),\
        F.col("new.additionalSettlement_amountTagType").alias("additionalSettlement_amountTagType"),\
        F.col("new.additionalSettlement_beneficiaryInstitutionCountryDescription").alias("additionalSettlement_beneficiaryInstitutionCountryDescription"),\
        F.col("new.additionalSettlement_beneficiaryInstitutionOneDescription").alias("additionalSettlement_beneficiaryInstitutionOneDescription"),\
        F.col("new.additionalSettlement_correspondentOneDescription").alias("additionalSettlement_correspondentOneDescription"),\
        F.col("new.additionalSettlement_debitOrCreditIndicator").alias("additionalSettlement_debitOrCreditIndicator"),\
        F.col("new.additionalSettlement_intermediaryOneDescription").alias("additionalSettlement_intermediaryOneDescription"),\
        F.col("new.additionalSettlement_orderingInstitutionOneDescription").alias("additionalSettlement_orderingInstitutionOneDescription"),\
        F.col("new.additionalSettlement_reimburseInstitutionOneDescription").alias("additionalSettlement_reimburseInstitutionOneDescription"),\
        F.col("new.additionalSettlement_settlementAmount").alias("additionalSettlement_settlementAmount"),\
        F.col("new.additionalSettlement_tagCurrencyDescription").alias("additionalSettlement_tagCurrencyDescription"))

In [0]:
## tx_jan_'+country
# TransactionReferenceCode? or 
# settlement_endtoEndTransactionReference

df_CAT_additionalsettlement = df_transaction_CAT_Hist.withColumn("new", F.arrays_zip('additionalSettlement_accountBranchName', 'additionalSettlement_accountBranch', 'additionalSettlement_accountCurrency', 'additionalSettlement_accountDescription', 'additionalSettlement_accountNumber', 'additionalSettlement_amountTagDescription', 'additionalSettlement_amountTagType', 'additionalSettlement_beneficiaryInstitutionCountryDescription', 'additionalSettlement_beneficiaryInstitutionOneDescription', 'additionalSettlement_correspondentOneDescription', 'additionalSettlement_debitOrCreditIndicator', 'additionalSettlement_intermediaryOneDescription', 'additionalSettlement_orderingInstitutionOneDescription', 'additionalSettlement_reimburseInstitutionOneDescription', 'additionalSettlement_settlementAmount', 'additionalSettlement_tagCurrencyDescription'))\
    .withColumn("new", F.explode("new"))\
    .select('TransactionReferenceCode', \
        F.col("new.additionalSettlement_accountBranchName").alias("additionalSettlement_accountBranchName"),\
        F.col("new.additionalSettlement_accountBranch").alias("additionalSettlement_accountBranch"),\
        F.col("new.additionalSettlement_accountCurrency").alias("additionalSettlement_accountCurrency"),\
        F.col("new.additionalSettlement_accountDescription").alias("additionalSettlement_accountDescription"),\
        F.col("new.additionalSettlement_accountNumber").alias("additionalSettlement_accountNumber"),\
        F.col("new.additionalSettlement_amountTagDescription").alias("additionalSettlement_amountTagDescription"),\
        F.col("new.additionalSettlement_amountTagType").alias("additionalSettlement_amountTagType"),\
        F.col("new.additionalSettlement_beneficiaryInstitutionCountryDescription").alias("additionalSettlement_beneficiaryInstitutionCountryDescription"),\
        F.col("new.additionalSettlement_beneficiaryInstitutionOneDescription").alias("additionalSettlement_beneficiaryInstitutionOneDescription"),\
        F.col("new.additionalSettlement_correspondentOneDescription").alias("additionalSettlement_correspondentOneDescription"),\
        F.col("new.additionalSettlement_debitOrCreditIndicator").alias("additionalSettlement_debitOrCreditIndicator"),\
        F.col("new.additionalSettlement_intermediaryOneDescription").alias("additionalSettlement_intermediaryOneDescription"),\
        F.col("new.additionalSettlement_orderingInstitutionOneDescription").alias("additionalSettlement_orderingInstitutionOneDescription"),\
        F.col("new.additionalSettlement_reimburseInstitutionOneDescription").alias("additionalSettlement_reimburseInstitutionOneDescription"),\
        F.col("new.additionalSettlement_settlementAmount").alias("additionalSettlement_settlementAmount"),\
        F.col("new.additionalSettlement_tagCurrencyDescription").alias("additionalSettlement_tagCurrencyDescription"))
